<table style="width: 100%; border-collapse: collapse; border: none; background: #fffbeb; border-left: 6px solid #f59e0b; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #78350f; font-size: 2em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        💡 01. Valores Faltantes: Estrategias de Imputación
      </h1>
      <p style="margin: 6px 0 0 0; color: #b45309; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Programación para Ciencia de Datos
      </p>
      <p style="margin: 4px 0 0 0; color: #92400e; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #f59e0b; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        💡 Para Dummies • Módulo 05
      </span><br>
      <span style="color: #78350f; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #b45309; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Data%20Science%20programming/05%20-%20Data%20Preparation/Para%20Dummies/01_Valores_Faltantes_Data_Preparation_Dummies.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## Los Valores Faltantes: El Mayor Dolor de Cabeza 🤕

Los valores faltantes (`NaN`) son inevitables. La pregunta no es **si** existen,
sino **cómo manejarlos** sin destruir la información.

### Estrategias Principales

| Estrategia | ¿Cuándo usarla? | Ventaja | Desventaja |
|---|---|---|---|
| **Eliminar filas** | Pocos nulos (<5%) | Simple | Pérdida de datos |
| **Imputar con media** | Variable numérica, sin outliers | Simple | Reduce varianza |
| **Imputar con mediana** | Variable numérica, con outliers | Robusta | Reduce varianza |
| **Imputar con moda** | Variable categórica | Razonable | Puede ser arbitrario |
| **Imputación KNN** | Cuando los patrones son importantes | Más precisa | Lenta en datos grandes |
| **Marcar como categoría** | 'Sin Dato' es información útil | Preserva info | Añade una categoría |


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Dataset con diferentes tipos de valores faltantes
np.random.seed(42)
n = 120
df = pd.DataFrame({
    'Temperatura': np.where(np.random.random(n) < 0.1, np.nan,  # 10% faltantes
                             np.random.normal(22, 4, n)),
    'Humedad': np.where(np.random.random(n) < 0.25, np.nan,    # 25% faltantes
                         np.random.uniform(40, 90, n)),
    'Velocidad_Viento': np.where(np.random.random(n) < 0.05, np.nan,  # 5% faltantes
                                  np.random.exponential(15, n)),
    'Condicion': np.where(np.random.random(n) < 0.15, np.nan,  # 15% faltantes
                           np.random.choice(['Soleado','Nublado','Lluvioso'], n))
})

print('📊 Reporte de Valores Faltantes:')
nulos = df.isnull().sum()
pct = (nulos / len(df) * 100).round(1)
reporte = pd.DataFrame({'Nulos': nulos, 'Porcentaje': pct, 'Estrategia': [
    'Mediana (robusta)', 'Mediana o KNN (muchos)', 'Eliminar filas (<5%)', 'Moda o nueva categoría'
]})
display(reporte)


In [ ]:
# Aplicando estrategias de imputación
df_imputado = df.copy()

# 1. Temperatura: imputar con la mediana
mediana_temp = df_imputado['Temperatura'].median()
df_imputado['Temperatura'] = df_imputado['Temperatura'].fillna(mediana_temp)
print(f'Temperatura — Mediana usada: {mediana_temp:.1f}°C')

# 2. Humedad: imputar con mediana
df_imputado['Humedad'] = df_imputado['Humedad'].fillna(df_imputado['Humedad'].median())

# 3. Velocidad del Viento: eliminar filas (solo 5% de nulos)
df_imputado = df_imputado.dropna(subset=['Velocidad_Viento']).reset_index(drop=True)

# 4. Condición: nueva categoría 'Desconocido'
df_imputado['Condicion'] = df_imputado['Condicion'].fillna('Desconocido')

print(f'\n✅ Nulos restantes: {df_imputado.isnull().sum().sum()}')
print(f'Filas: {len(df)} → {len(df_imputado)} (se eliminaron {len(df)-len(df_imputado)} filas)')
print(f'Distribución de Condición:')
print(df_imputado['Condicion'].value_counts())


In [ ]:
# Visualización: Antes vs Después
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Temperatura original (con NaN)
df['Temperatura'].hist(bins=20, ax=axes[0], color='#ef4444', edgecolor='white', alpha=0.7)
axes[0].set_title(f'Temperatura ORIGINAL\n({df["Temperatura"].isnull().sum()} NaN)',
                   fontweight='bold')
axes[0].set_xlabel('°C')

# Temperatura imputada
df_imputado['Temperatura'].hist(bins=20, ax=axes[1], color='#22c55e', edgecolor='white', alpha=0.7)
axes[1].axvline(mediana_temp, color='red', linewidth=2, linestyle='--',
                 label=f'Mediana: {mediana_temp:.1f}°C')
axes[1].set_title('Temperatura IMPUTADA\n(sin NaN)', fontweight='bold')
axes[1].set_xlabel('°C')
axes[1].legend()

plt.tight_layout()
plt.show()


---
### ✅ Autocomprobación Rápida

**Pregunta:** Una columna de 'Ingresos Anuales' tiene el 30% de valores faltantes. ¿Qué estrategia NO deberías usar y por qué?

<details>
<summary>💡 Ver respuesta</summary>

**No deberías eliminar las filas** con ingresos faltantes.

Con el 30% de los datos faltantes, eliminar esas filas causaría:
1. **Pérdida masiva de datos** (perdemos el 30% del dataset)
2. **Sesgo de selección**: Los ingresos suelen faltar por razones específicas
   (ej: personas de bajos ingresos podrían evitar declararlos).

Mejor alternativa: imputación con **mediana** (robusta a outliers en salarios),
o imputación con **KNN** que usa otras variables correlacionadas.

</details>


---
<div align="center">
  <p style="font-size: 0.9em; color: #64748b;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Programación para Ciencia de Datos (Edición Para No Ingenieros)</i>
  </p>
</div>
